# UD5. El modelo roto

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**
Material de la **parte 3 de la P5.3** · Criterio **2.e**

---

Este cuaderno entrena un clasificador sobre Fashion-MNIST y obtiene un resultado muy por
debajo de lo razonable. Un compañero te lo pasa y te pide ayuda.

**Tiene tres errores sembrados a propósito**, de tres tipos distintos:

| | Tipo |
|---|---|
| 1 | Se detecta con una de las siete comprobaciones de sensatez |
| 2 | Solo se ve mirando las curvas de aprendizaje |
| 3 | No se ve en ningún sitio hasta que se comparan dos cifras que deberían coincidir |

Ninguno de los tres lanza una excepción. El cuaderno se ejecuta entero, imprime números y
dibuja figuras, y todo parece estar bien si no se comprueba.

## Lo que hay que hacer

**No busques los tres a la vez.** El método es el del bloque 16 de los apuntes: aplicar las
comprobaciones en orden y parar en la primera que falle. Para cada error, entrega:

| | |
|---|---|
| El síntoma | qué se observa, con la cifra |
| La comprobación que lo pilla | cuál de las siete, o cuál inventaste |
| La causa | la línea exacta |
| El arreglo | y la cifra después de arreglarlo |
| **Por qué no daba error** | esta es la parte importante |

Y al final, la exactitud con los tres arreglados, comparada con la de este cuaderno y con
la de la clase mayoritaria.

> **Documenta el orden en el que los encontraste**, aunque no sea el orden en el que están
> numerados arriba. Ese orden es la parte del ejercicio que demuestra el método.

## Lo que NO hay que hacer

- Reescribir el cuaderno desde cero. El ejercicio es diagnosticar, no sustituir.
- Tocar la arquitectura, el número de épocas o el tamaño de lote. **Los tres errores están
  fuera de la arquitectura**, y cambiarla solo enturbia la medición.
- Buscar en el historial de git ni preguntar al profesorado cuáles son.

---

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import matplotlib.pyplot as plt
import keras

keras.utils.set_random_seed(20262027)

CLASES = ["camiseta", "pantalon", "jersey", "vestido", "abrigo",
          "sandalia", "camisa", "zapatilla", "bolso", "botin"]

(X_todo, y_todo), (X_prueba, y_prueba) = keras.datasets.fashion_mnist.load_data()
print("datos cargados:", X_todo.shape, X_prueba.shape)

## 1. Preparar los datos

In [ ]:
# Entrenamiento: las primeras 10.000 imagenes, escaladas al rango [0, 1].
X_entrena = X_todo[:10000].astype("float32") / 255.0
y_entrena = y_todo[:10000]

# Validacion: para vigilar el entrenamiento sin tocar la particion de prueba.
X_valida = X_entrena[:2000]
y_valida = y_entrena[:2000]

# Prueba: el conjunto de prueba oficial de Fashion-MNIST.
X_test = X_prueba.astype("float32")
y_test = y_prueba

print(f"entrenamiento {X_entrena.shape}")
print(f"validacion    {X_valida.shape}")
print(f"prueba        {X_test.shape}")

## 2. El modelo

In [ ]:
modelo = keras.Sequential([
    keras.layers.Input(shape=(28, 28)),
    keras.layers.Flatten(),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])

modelo.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.08),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

modelo.summary()

## 3. Entrenar

In [ ]:
historia = modelo.fit(
    X_entrena, y_entrena,
    epochs=25,
    batch_size=64,
    validation_data=(X_valida, y_valida),
    verbose=0,
)

print(f"perdida final de entrenamiento: {historia.history['loss'][-1]:.4f}")
print(f"perdida final de validacion:    {historia.history['val_loss'][-1]:.4f}")
print(f"exactitud final de validacion:  {historia.history['val_accuracy'][-1]:.4f}")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4.2))
ejes[0].plot(historia.history["loss"], label="entrenamiento")
ejes[0].plot(historia.history["val_loss"], label="validacion")
ejes[0].set_xlabel("epoca");  ejes[0].set_ylabel("entropia cruzada")
ejes[0].legend()
ejes[1].plot(historia.history["accuracy"], label="entrenamiento")
ejes[1].plot(historia.history["val_accuracy"], label="validacion")
ejes[1].set_xlabel("epoca");  ejes[1].set_ylabel("exactitud")
ejes[1].legend()
fig.suptitle("Curvas de aprendizaje")
fig.tight_layout()
plt.show()

## 4. Evaluar

In [ ]:
perdida, exactitud = modelo.evaluate(X_test, y_test, verbose=0)
print(f"exactitud sobre el conjunto de prueba: {exactitud:.4f}")
print()
print("Y eso es mucho menos de lo que se esperaba de Fashion-MNIST con una red")
print("densa de este tamaño. Algo va mal, y no hay ningun mensaje de error.")

In [ ]:
pred = modelo.predict(X_test, verbose=0).argmax(axis=1)

fig, ejes = plt.subplots(2, 5, figsize=(11, 5))
rng = np.random.default_rng(0)
for eje, idx in zip(ejes.ravel(), rng.choice(len(X_test), 10, replace=False)):
    eje.imshow(X_test[idx], cmap="gray")
    eje.set_title(f"es {CLASES[y_test[idx]]}\ndice {CLASES[pred[idx]]}", fontsize=8)
    eje.axis("off")
fig.suptitle("Diez predicciones sobre el conjunto de prueba", y=1.02)
fig.tight_layout()
plt.show()

---

## Por dónde empezar

Las siete comprobaciones del bloque 16.2, en orden:

| # | Comprobación |
|---|---|
| 1 | Mirar los datos: veinte ejemplos con su etiqueta |
| 2 | Reparto de clases, y exactitud de la clase mayoritaria |
| 3 | Rango, tipo y forma de `X` — **en las tres particiones** |
| 4 | Formas de entrada y de salida |
| 5 | Pérdida inicial $= \log(k)$ |
| 6 | Sobreajustar 20 muestras |
| 7 | El punto de referencia |

Y la tabla de diagnóstico rápido:

| Síntoma | Causa más probable |
|---|---|
| Pérdida `NaN` desde el principio | tasa de aprendizaje alta, o un `log(0)`, o `NaN` en los datos |
| Pérdida clavada en $\log(k)$ | etiquetas mal, o la última capa sin activación |
| Exactitud igual a la de la clase mayoritaria | conjunto desequilibrado y modelo que contesta siempre lo mismo |
| Entrenamiento perfecto, validación mala | sobreajuste |
| Validación mejor que entrenamiento | dropout, o partición de validación demasiado fácil |
| Resultados magníficos e increíbles | fuga de información |
| La validación va bien y la prueba mal | **las dos particiones no miden lo mismo** |

Esa última fila no estaba en los apuntes, y es la pista del tercer error.